Structured output


In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:llama-3.1-8b-instant")


In [4]:
from pydantic import BaseModel, Field

class movie(BaseModel):
    title: str= Field(description="The title of the movie")
    year:int= Field(description="The year the movie was released")
    rating:float= Field(description="The rating of the movie out of 10")
    director:str= Field(description="The director of the movie")


In [5]:
model_with_structure=model.with_structured_output(movie)
model_with_structure



_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A65919B620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A6593A81A0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'prop

In [8]:
response=model_with_structure.invoke("give me the details of the movie Avengers")
response

movie(title='Avengers', year=2012, rating=7.5, director='Joss Whedon')

In [9]:
from pydantic import BaseModel, Field

class movie(BaseModel):
    title: str= Field(..., description="The title of the movie")
    year:int= Field(..., description="The year the movie was released")
    rating:float= Field(..., description="The rating of the movie out of 10")
    director:str= Field(..., description="The director of the movie")

model_with_structure=model.with_structured_output(movie,include_raw=True)

response=model_with_structure.invoke("give me the details of the movie Avengers Endgame")
response


{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'x9gcqm6kf', 'function': {'arguments': '{"director":"Anthony and Joe Russo","rating":8.7,"title":"Avengers: Endgame","year":2019}', 'name': 'movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 284, 'total_tokens': 322, 'completion_time': 0.038918817, 'completion_tokens_details': None, 'prompt_time': 0.015902965, 'prompt_tokens_details': None, 'queue_time': 0.048697865, 'total_time': 0.054821782}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee52d-3ce4-7e83-9daa-b2c96c59058b-0', tool_calls=[{'name': 'movie', 'args': {'director': 'Anthony and Joe Russo', 'rating': 8.7, 'title': 'Avengers: Endgame', 'year': 2019}, 'id': 'x9gcqm6kf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284,

### Nested structure


In [10]:
from pydantic import BaseModel, Field

class actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[actor]
    rating:float
    genres:list[str]
    budget:float | None=Field(default=None, description="The budget of the movie in USD")

model_with_structure=model.with_structured_output(MovieDetails,include_raw=True)
response=model_with_structure.invoke("give me the details of the movie Avengers Endgame")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hdrtc283k', 'function': {'arguments': '{"budget":356000000,"cast":[{"name":"Robert Downey Jr.","role":"Tony Stark"},{"name":"Chris Evans","role":"Steve Rogers"},{"name":"Mark Ruffalo","role":"Bruce Banner"},{"name":"Chris Hemsworth","role":"Thor"},{"name":"Scarlett Johansson","role":"Natasha Romanoff"},{"name":"Jeremy Renner","role":"Clint Barton"},{"name":"Brie Larson","role":"Carol Danvers"},{"name":"Paul Rudd","role":"Scott Lang"},{"name":"Don Cheadle","role":"James \'Rhodey\' Rhodes"},{"name":"Karen Gillan","role":"Nebula"},{"name":"Danai Gurira","role":"Okoye"},{"name":"Bradley Cooper","role":"Sam Wilson"},{"name":"Gwyneth Paltrow","role":"Pepper Potts"},{"name":"Jon Favreau","role":"Happy Hogan"},{"name":"Josh Brolin","role":"Thanos"}],"genres":["Action","Adventure","Sci-Fi"],"rating":8.7,"title":"Avengers: Endgame","year":2019}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'token_usage

### Type Dict

In [19]:
from typing_extensions import Annotated,TypedDict

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str, ..., "The title of the movie"]
    year:Annotated[int, ..., "The year the movie was released"]
    rating:Annotated[float, ..., "The rating of the movie out of 10"]
    director:Annotated[str, ..., "The director of the movie"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide me the detail of the movie Avengers endgame")
response

{'director': 'Anthony and Joe Russo',
 'rating': 8,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [20]:
class actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[actor]
    rating:float
    genres:list[str]
    budget:float | None=Field(default=None, description="The budget of the movie in USD")

model_with_structure=model.with_structured_output(MovieDetails,include_raw=True)
response=model_with_structure.invoke("give me the details of the movie Avengers Endgame")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm878kd5r3', 'function': {'arguments': '{"budget":356000000,"cast":[{"name":"Robert Downey Jr.","role":"Tony Stark"},{"name":"Chris Evans","role":"Steve Rogers"},{"name":"Mark Ruffalo","role":"Bruce Banner"},{"name":"Chris Hemsworth","role":"Thor"},{"name":"Scarlett Johansson","role":"Natasha Romanoff"},{"name":"Jeremy Renner","role":"Clint Barton"},{"name":"Brie Larson","role":"Carol Danvers"}],"genres":["Action","Adventure","Fantasy","Science Fiction","Thriller"],"name":"Avengers Endgame","rating":8.7,"title":"Avengers: Endgame","year":2019}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 628, 'total_tokens': 809, 'completion_time': 0.255836245, 'completion_tokens_details': None, 'prompt_time': 0.038178433, 'prompt_tokens_details': None, 'queue_time': 0.056943977, 'total_time': 0.294014678}, 'model_name': 'llama-3.1-8b-instant', 'syste

In [22]:
model.profile

{'name': 'Llama 3.1 8B Instant',
 'release_date': '2024-07-23',
 'last_updated': '2024-07-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 131072,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}